# 19. Multi-faceted Filtering

Layer metadata filters, a score threshold and diversity (MMR) on top of vector search.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Real systems rarely rank by similarity alone. **Multi-faceted filtering** layers several signals: hard *metadata* filters (section, tenant, date, permissions), a *score threshold* to drop weak matches, and *diversity* (MMR) so results aren't near-duplicates. Metadata filtering is also how you enforce access control at retrieval time.

## How it works

1. Attach metadata (section, word count...) to each chunk
2. Apply a boolean mask for allowed values
3. Drop results below a similarity threshold
4. Re-rank with MMR to balance relevance and novelty
5. Pass the survivors to the LLM

```text
 all chunks --> [metadata filter] --> [score threshold] --> [MMR diversity] --> k results
 (permissions, section, date)        (drop weak matches)    (avoid duplicates)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import numpy as np

chunks, meta = [], []
for heading, body in split_sections(load_text()):
    for c in chunk_text(body, size=400, overlap=40):
        chunks.append(c)
        meta.append({"section": heading, "words": len(c.split())})
store = build_store(chunks, meta)
print(sorted({m["section"] for m in meta}))

### Facet 1: metadata filter
Restrict candidates by section (this is also how you enforce per-user access control).

In [ ]:
def search_filtered(query, sections=None, min_score=0.25, k=5):
    mask = np.array([sections is None or m["section"] in sections for m in meta])
    return [h for h in store.search(query, k * 3, mask=mask) if h["score"] >= min_score][:k]

query = "How do I reduce hallucinations?"
show(search_filtered(query, sections={"Hallucinations and Grounding", "Evaluating RAG Systems"}), width=100)

### Facet 2: diversity with MMR
Maximal Marginal Relevance picks results that are relevant **and** different from each other.

In [ ]:
def mmr(query, k=4, lam=0.6, pool=15):
    q = embed(query)[0]
    cand = [h["idx"] for h in store.search(query, pool)]
    chosen = []
    while cand and len(chosen) < k:
        def score(i):
            rel = store.vecs[i] @ q
            red = max((store.vecs[i] @ store.vecs[j] for j in chosen), default=0)
            return lam * rel - (1 - lam) * red
        best = max(cand, key=score)
        chosen.append(best); cand.remove(best)
    return [{"idx": i, "text": chunks[i], "score": float(store.vecs[i] @ q), "meta": meta[i]} for i in chosen]

hits = mmr("What affects RAG quality and speed?")
for h in hits: print(h["meta"]["section"], f"{h['score']:.2f}")
print(answer("What affects RAG quality and speed?", [h["text"] for h in hits]))

## Effect of each facet
Track how many candidates survive each stage.

In [ ]:
q = "How do I make a RAG system fast and secure?"
all_hits = store.search(q, len(chunks))
step1 = [h for h in all_hits if meta[h["idx"]]["section"] in {"Caching and Latency", "Security and Privacy", "Hybrid Search and BM25"}]
step2 = [h for h in step1 if h["score"] >= 0.25]
print(f"all chunks: {len(all_hits)} -> after metadata filter: {len(step1)} -> after threshold: {len(step2)}")

## MMR vs plain top-k: redundancy
Measure how similar the returned chunks are to each other (lower = more diverse).

In [ ]:
import numpy as np
def redundancy(hits):
    v = store.vecs[[h["idx"] for h in hits]]
    sim = v @ v.T
    return float((sim.sum() - len(hits)) / (len(hits) * (len(hits) - 1)))

q = "What affects RAG quality and speed?"
print("top-k redundancy:", round(redundancy(store.search(q, 5)), 3))
for lam in [1.0, 0.7, 0.5, 0.3]:
    print(f"MMR lambda={lam}: redundancy", round(redundancy(mmr(q, k=5, lam=lam)), 3),
          "sections:", sorted({h["meta"]["section"][:18] for h in mmr(q, k=5, lam=lam)}))

## Evaluate retrieval quality

In [ ]:
compare_retrievers({"top-k": store.search, "MMR": lambda q, k=3: mmr(q, k=k), "threshold 0.3": lambda q, k=3: search_filtered(q, min_score=0.3, k=k)}, k=3)

## Access control demo
The same query returns different chunks for different users when the mask is built from their permissions.

In [ ]:
user_access = {"alice (public docs)": {"Introduction to RAG", "Embeddings", "Vector Databases"}, "bob (security team)": {"Security and Privacy", "Hallucinations and Grounding"}}
for user, allowed in user_access.items():
    print(user, "->", [(h["meta"]["section"]) for h in search_filtered("how do I protect data?", sections=allowed, min_score=0.0, k=2)])

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Enforces permissions and scoping | Needs good metadata hygiene |
| Removes weak matches and duplicates | Thresholds are corpus- and model-specific |
| Composable: each facet is independent | MMR adds compute on large pools |

## When to use it

Multi-tenant or regulated data, large mixed corpora, and any time results look repetitive.

## Experiments to try

- Add a date field and filter by recency.
- Calibrate the threshold from the score distribution of golden questions.
- Make `lam` depend on the question type.

## Key takeaways

- Filter by metadata before ranking.
- Thresholds prevent junk context.
- MMR trades relevance for diversity.

---
**Next:** [20. Hierarchical Indices](../20_hierarchical_indices/hierarchical_indices.ipynb)